### 1. 自然语言处理中，语言模型经历了从统计到神经网络的模型演进。

- 请使用本章提供的迷你语料库（`datawhale agent learns`，`datawhale agent works`），计算句子 `agent works` 在 Bigram 模型下的概率
- N-gram 模型的核心假设是马尔可夫假设。请解释这个假设的含义，以及 N-gram 模型存在哪些根本性局限？
- 神经网络语言模型（RNN/LSTM）和 Transformer 分别是如何克服 N-gram 模型局限的？它们各自的优势是什么？

In [1]:
import collections

# 语料库
corpus = "datawhale agent learns datawhale agent works"
tokens = corpus.split()
total_tokens = len(tokens)

# --- 第一步: 计算 P(agent) ---
count_agent = tokens.count('agent')
p_agent = count_agent / total_tokens
print(f"第一步: P(agent) = {count_agent}/{total_tokens} = {p_agent:.3f}")

# --- 第二步: 计算 P(works|agent) ---
bigrams = zip(tokens, tokens[1:])
bigram_counts = collections.Counter(bigrams)
count_agent_works = bigram_counts[('agent', 'works')]
p_works_given_agent = count_agent_works / count_agent
print(f"第二步: P(works|agent) = {count_agent_works}/{count_agent} = {p_works_given_agent:.3f}")

# --- 最后: 将概率连乘 ---
p_sentence = p_agent * p_works_given_agent
print(f"最后: P('agent works') ≈ {p_agent:.3f} * {p_works_given_agent:.3f} = {p_sentence:.3f}")

第一步: P(agent) = 2/6 = 0.333
第二步: P(works|agent) = 1/2 = 0.500
最后: P('agent works') ≈ 0.333 * 0.500 = 0.167


### 1. 马尔可夫假设的含义及 N-gram 模型的根本性局限

**马尔可夫假设的含义：**  
一个词出现的概率**只依赖于它前面有限的 $N-1$ 个词**，而与更早的历史上下文无关，从而简化联合概率计算：

$$P(w_t \mid w_1, \dots, w_{t-1}) \approx P(w_t \mid w_{t-N+1}, \dots, w_{t-1})$$

**N-gram 模型的根本性局限：**
- **长距离依赖缺失**：上下文窗口被固定截断在 $N-1$ 个词，无法捕捉长句中远距离的语法与语义关联。
- **数据稀疏与零概率问题**：语料库无法穷举所有词组，一旦出现未见过的词组合，连乘概率就会直接归零（需额外依赖平滑技术）。
- **维度灾难**：词表大小为 $|V|$ 时，参数空间随 $N$ 呈指数级膨胀（$O(|V|^N)$），增大 $N$ 会导致存储与计算成本失控。
- **缺乏语义泛化能力**：将词视为孤立的离散符号，仅做字面计数，无法将学到的规律泛化到近义词上（如见过 `agent works` 却无法推断 `robot works`）。

---

### 2. RNN/LSTM 与 Transformer 如何克服局限及各自优势

两者均先通过**词嵌入（Word Embedding）**将离散词映射到低维连续向量空间，使语义相近的词距离更近，从根本上缓解了**语义泛化缺失、数据稀疏和维度灾难**问题。在此基础上，两者以不同架构解决了长上下文建模难题：

#### RNN / LSTM
- **如何克服局限**：通过循环传递的**隐藏状态（Hidden State）**逐词累积历史信息，打破固定的 $N-1$ 窗口限制；**LSTM** 进一步引入遗忘门、输入门、输出门及细胞状态，有效缓解长序列梯度消失问题，记住更长的关键依赖。
- **核心优势**：
  - 天然适配变长序列，按时间步推进的结构自带时序感知能力。
  - 参数在时间步间共享，推理时只需维护固定大小的隐藏状态，单步计算与显存开销恒定。

#### Transformer
- **如何克服局限**：摒弃循环结构，采用**自注意力机制（Self-Attention）**直接计算序列中任意两个词之间的相关性（信息传递路径长度为 $O(1)$），并辅以**位置编码（Positional Encoding）**保留词序，彻底解决信息逐步衰减的问题。
- **核心优势**：
  - **极高的并行计算效率**：训练时可对整个序列同时做矩阵运算，打破了 RNN 必须逐词串行计算的瓶颈。
  - **卓越的长距离建模与可扩展性**：多头注意力能精准捕捉全局复杂语义，且架构易于堆叠至海量参数规模（符合 Scaling Law），成为现代大语言模型的基石。

### 2. Transformer架构<sup>[4]</sup>是现代大语言模型的基础。其中：

> **提示：** 可以结合本章 3.1.2 节的代码实现来辅助理解

- 自注意力机制（Self-Attention）的核心思想是什么？
- 为什么 Transformer 能够并行处理序列，而 RNN 必须串行处理？位置编码（Positional Encoding）在其中起什么作用？
- Decoder-Only 架构与完整的 Encoder-Decoder 架构有什么区别？为什么现在主流的大语言模型都采用 Decoder-Only 架构？

#### 自注意力机制（Self-Attention）的核心思想
让序列中的每个 Token 动态计算与序列中**所有其他 Token** 的相关性权重，并据此对信息进行加权聚合，从而直接捕获全局上下文语义与长距离依赖。
- **计算本质**：将输入映射为查询（$Q$）、键（$K$）、值（$V$），通过缩放点积计算相关性并加权求和：
  $$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

#### 并行处理能力与位置编码（Positional Encoding）的作用
- **为什么 Transformer 能并行而 RNN 必须串行**：
  - **RNN**：当前隐藏状态 $h_t$ 强依赖于上一时刻的状态 $h_{t-1}$，存在时间步上的先后依赖，只能逐步串行计算。
  - **Transformer**：摒弃了循环结构，通过矩阵乘法一次性计算序列内所有位置两两之间的注意力，各位置间无计算顺序依赖，因此训练阶段可高度并行。
- **位置编码的作用**：自注意力机制本身具有**置换不变性**（不感知输入顺序，等价于无序集合）。位置编码通过向词向量中注入绝对或相对位置信息（如正余弦编码、RoPE 旋转位置编码），赋予模型感知词序和语序结构的能力。

#### Decoder-Only 与 Encoder-Decoder 的区别及主流选择原因
- **核心区别**：
  - **Encoder-Decoder**：编码器使用双向注意力充分理解输入，解码器使用单向因果掩码（Causal Mask）+ 交叉注意力进行条件生成，适合翻译、摘要等典型 Seq2Seq 任务。
  - **Decoder-Only**：仅保留解码器并使用单向因果掩码（每个 Token 只能看到自己及之前的 Token），将所有任务统一为自回归的“预测下一个 Token（Next-Token Prediction）”。
- **主流 LLM 采用 Decoder-Only 的原因**：
  1. **任务高度统一，泛化更强**：问答、推理、翻译、代码等任务均可统一为文本续写，天然适配 Prompt 与上下文学习（In-Context Learning）。
  2. **训练效率与扩展性（Scaling Law）更优**：无 Encoder-Decoder 间的信息压缩瓶颈，结构更简洁；在无监督预训练中所有 Token 都参与损失计算，数据与算力利用率更高。
  3. **推理加速友好**：单向注意力天然支持 **KV Cache** 复用，大幅降低长文本和多轮对话的推理延迟。

### 3. 文本子词分词算法是大语言模型的一项关键技术，负责将文本转换为模型可处理的 token 序列。那为什么不能直接以"字符"或"单词"作为模型的输入单元？BPE（Byte Pair Encoding）算法解决了什么问题？

#### 为什么不直接以“单词”或“字符”作为输入单元？
- **按单词（Word-level）切分的弊端**：
  1. **词表爆炸**：语言词汇及形态变化极多（如 `run / running / ran`），导致 Embedding 层和输出层参数量与计算开销剧增。
  2. **OOV（未登录词）问题**：遇到训练集外的新词、拼写错误或专有名词时只能退化为 `<UNK>`，丢失语义。
  3. **难以共享词形语义**：无法捕捉词根、词缀间的关联（如 `happy` 与 `unhappy` 被当作完全独立的词）。
- **按字符（Character-level）切分的弊端**：
  1. **序列过长、计算昂贵**：同一文本拆分后 Token 数暴增，而自注意力复杂度为 $O(n^2)$，会显著拉高显存与算力开销，并压缩有效上下文长度。
  2. **语义粒度过细**：单个字符缺乏独立语义，模型需要额外消耗大量参数去学习拼写规则与词级含义。

#### BPE（Byte Pair Encoding）算法解决了什么问题？
BPE 在**词表大小**与**序列长度**之间取得了平衡，并有效解决了 **OOV 问题**：
- **核心机制**：从基础字符（或字节，即 Byte-level BPE）起步，统计语料中相邻符号对的出现频率，迭代合并最高频的符号对，直到达到预设词表规模。
- **实际效果**：**高频词**被合并为完整 Token（压缩序列长度、保留完整语义），**低频词与生僻词**被拆分为常见子词或词根（消除 `<UNK>`、提升模型对新词和词形变化的泛化能力）。

In [2]:
import re
from collections import Counter, defaultdict

class SimpleBPE:
    def __init__(self, num_merges=10):
        self.num_merges = num_merges
        self.merges = []          # 按学习顺序记录合并规则: [(a, b), ...]
        self.merge_ranks = {}     # 规则优先级字典: {(a, b): rank}
        self.token2id = {}        # token -> id
        self.id2token = {}        # id -> token
        self.unk_token = "<unk>"
        self.eow_token = "</w>"   # 词尾结束标记

    def _build_initial_vocab(self, corpus):
        """将原始文本语料转为 {'h u g </w>': freq} 的格式"""
        word_freqs = Counter()
        for text in corpus:
            for word in text.strip().split():
                word_freqs[word] += 1

        vocab = {}
        for word, freq in word_freqs.items():
            # 将每个词拆分为字符并用空格隔开，末尾加上 </w>
            chars = " ".join(list(word)) + f" {self.eow_token}"
            vocab[chars] = freq
        return vocab

    @staticmethod
    def _get_stats(vocab):
        """统计相邻词元对频率（与参考代码一致）"""
        pairs = defaultdict(int)
        for word, freq in vocab.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i + 1])] += freq
        return pairs

    @staticmethod
    def _merge_vocab(pair, v_in):
        """合并指定的词元对（与参考代码一致）"""
        v_out = {}
        bigram = re.escape(" ".join(pair))
        p = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
        merged = "".join(pair)
        for word, freq in v_in.items():
            w_out = p.sub(lambda _: merged, word)
            v_out[w_out] = freq
        return v_out

    def train(self, corpus, verbose=True):
        """训练 BPE：学习合并规则并构建词表"""
        vocab = self._build_initial_vocab(corpus)

        # 收集基础字符集，避免合并后丢失单个基础字符的映射
        tokens = set()
        for word in vocab:
            tokens.update(word.split())

        self.merges = []
        for i in range(self.num_merges):
            pairs = self._get_stats(vocab)
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            vocab = self._merge_vocab(best, vocab)
            self.merges.append(best)
            tokens.add("".join(best))

            if verbose:
                print(f"第{i + 1}次合并: {best} -> {''.join(best)} (频次: {pairs[best]})")
                print(f"当前词表切分: {list(vocab.keys())}")
                print("-" * 40)

        # 记录合并优先级（越早学到的规则优先级越高）
        self.merge_ranks = {pair: idx for idx, pair in enumerate(self.merges)}

        # 构建 token <-> id 映射表
        sorted_tokens = [self.unk_token] + sorted(tokens)
        self.token2id = {tok: idx for idx, tok in enumerate(sorted_tokens)}
        self.id2token = {idx: tok for tok, idx in self.token2id.items()}

    def _tokenize_word(self, word):
        """按训练时学到的优先级，对单个词执行 BPE 合并"""
        symbols = list(word) + [self.eow_token]
        while len(symbols) >= 2:
            pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            # 找到当前词中优先级最高（rank 最小）的可合并对
            best_pair = min(pairs, key=lambda p: self.merge_ranks.get(p, float("inf")))
            if best_pair not in self.merge_ranks:
                break

            # 从左到右合并所有不重叠的 best_pair
            new_symbols = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best_pair:
                    new_symbols.append(symbols[i] + symbols[i + 1])
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols
        return symbols

    def tokenize(self, text):
        """将输入文本切分为 BPE token 列表"""
        tokens = []
        for word in text.strip().split():
            tokens.extend(self._tokenize_word(word))
        return tokens

    def encode(self, text):
        """将文本编码为 token ID 列表"""
        tokens = self.tokenize(text)
        unk_id = self.token2id[self.unk_token]
        return [self.token2id.get(tok, unk_id) for tok in tokens]

    def decode(self, ids):
        """将 token ID 列表还原为文本"""
        tokens = [self.id2token.get(idx, self.unk_token) for idx in ids]
        return "".join(tokens).replace(self.eow_token, " ").strip()


# ==================== 运行演示 ====================

# 1. 准备与参考代码等价的语料并训练
corpus = ["hug pug pun bun"]
bpe = SimpleBPE(num_merges=4)
bpe.train(corpus, verbose=True)

print("最终词表 (token2id):")
print(bpe.token2id)
print("=" * 40)

# 2. 测试编码与解码（包含未在语料中出现过的新词 "bug" 和含未知字符的 "mug"）
for test_text in ["hug pun", "bug bun", "mug"]:
    tokens = bpe.tokenize(test_text)
    ids = bpe.encode(test_text)
    decoded = bpe.decode(ids)
    print(f"原文  : {test_text}")
    print(f"切分  : {tokens}")
    print(f"编码ID: {ids}")
    print(f"解码  : {decoded}")
    print("-" * 40)

第1次合并: ('u', 'g') -> ug (频次: 2)
当前词表切分: ['h ug </w>', 'p ug </w>', 'p u n </w>', 'b u n </w>']
----------------------------------------
第2次合并: ('ug', '</w>') -> ug</w> (频次: 2)
当前词表切分: ['h ug</w>', 'p ug</w>', 'p u n </w>', 'b u n </w>']
----------------------------------------
第3次合并: ('u', 'n') -> un (频次: 2)
当前词表切分: ['h ug</w>', 'p ug</w>', 'p un </w>', 'b un </w>']
----------------------------------------
第4次合并: ('un', '</w>') -> un</w> (频次: 2)
当前词表切分: ['h ug</w>', 'p ug</w>', 'p un</w>', 'b un</w>']
----------------------------------------
最终词表 (token2id):
{'<unk>': 0, '</w>': 1, 'b': 2, 'g': 3, 'h': 4, 'n': 5, 'p': 6, 'u': 7, 'ug': 8, 'ug</w>': 9, 'un': 10, 'un</w>': 11}
原文  : hug pun
切分  : ['h', 'ug</w>', 'p', 'un</w>']
编码ID: [4, 9, 6, 11]
解码  : hug pun
----------------------------------------
原文  : bug bun
切分  : ['b', 'ug</w>', 'b', 'un</w>']
编码ID: [2, 9, 2, 11]
解码  : bug bun
----------------------------------------
原文  : mug
切分  : ['m', 'ug</w>']
编码ID: [0, 9]
解码  : <unk>ug
------

### 4. 本章 3.2.3 节介绍了如何本地部署开源大语言模型。请完成以下实践和分析：

> **提示：** 这是一道动手实践题，建议实际操作

- 按照本章的指导，在本地部署一个轻量级的开源模型（推荐 `Qwen3-0.6B`），并尝试调整采样参数并观察其对输出的影响
- 选择一个具体任务（如文本分类、信息抽取、代码生成等），设计并对比以下不同的提示策略（如 Zero-shot、Few-shot、Chain-of-Thought）对输出结果的效果差异
- 从性能、成本、可控性、隐私等维度比较闭源模型和开源模型
- 如果你要构建一个企业级的客服智能体，你会选择哪种类型的模型？需要考虑哪些因素？

性能上普遍来说开源模型都要弱于闭源模型，闭源像gemini，gpt，claude；开源里面最强就kimi k3.1 pro。成本的话目前都是买token，付费订阅或者调api。本地部署可行性上开源模型更适配。可控性做自己这个领域的微调肯定是开源模型，像deepseek的ds harness以及workbuddy调hy4，隐私最强肯定是本地部署的模型。

我会选择本地部署的开源模型，公司数据隐私要求高，客服要求响应速度要快，而且说的一定要准确，要有置信度，不确定的就丢给人工客服。而且总的成本也不能太高。通过rag去链接企业的外部知识库。

In [1]:
import os

# 1. 将模型缓存与下载路径强制指定到 D 盘（必须在导入 transformers 前执行）
os.environ["HF_HOME"] = r"D:\hf_cache"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 指定模型 ID（若旧版 transformers 报错，可先 pip install -U transformers 或换用 "Qwen/Qwen3-0.6B"）
model_id = "Qwen/Qwen3.5-0.8B"

# 检查 GPU 设备
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU 型号: {torch.cuda.get_device_name(0)}")

# 加载分词器与模型（5060 原生支持 bfloat16，显存占用减半且推理更快）
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32
).to(device)
model.eval()

print(f"模型加载完成！当前已存放在: {os.environ['HF_HOME']}")
if device == "cuda":
    print(f"当前显存占用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Using device: cuda
GPU 型号: NVIDIA GeForce RTX 5060 Laptop GPU


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

d:\miniconda\envs\pytorch\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\hf_cache\hub\models--Qwen--Qwen3.5-0.8B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

d:\miniconda\envs\pytorch\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\hf_cache\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

... (4 outputs left out)

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

模型加载完成！当前已存放在: D:\hf_cache
当前显存占用: 1.42 GB


In [2]:
def chat(
    prompt,
    system_prompt="You are a helpful assistant.",
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    repetition_penalty=1.0,
    enable_thinking=False  # 关闭内置思考模式，便于清晰观察 CoT 与采样参数效果
):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    # 兼容支持/不支持 enable_thinking 参数的分词器模板
    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    # 组装生成参数（贪心解码时不传入 temperature/top_p/top_k 以免触发警告）
    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "repetition_penalty": repetition_penalty,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        gen_kwargs.update({
            "temperature": temperature,
            "top_p": top_p,
            "top_k": top_k,
        })

    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            **gen_kwargs
        )

    # 截取掉输入部分的 Token，只保留新生成的内容
    new_tokens = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
    return response.strip()

# 基础连通性测试
print("【基础测试回答】")
print(chat("你好，请用一句话介绍你自己。"))

【基础测试回答】


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


你好！我是 Qwen3.5，阿里巴巴通义实验室研发的超大规模语言模型，能够理解、推理、生成和创作各种内容，致力于为用户提供高质量、安全、有帮助的对话体验。


In [3]:
test_prompt = "请为一家主打深夜营业的AI主题咖啡馆写3句不同风格的宣传标语。"

configs = [
    {"name": "1. 贪心解码 (确定性输出)", "do_sample": False, "repetition_penalty": 1.0},
    {"name": "2. 低温度保守 (T=0.2, top_p=0.5)", "do_sample": True, "temperature": 0.2, "top_p": 0.5, "top_k": 20},
    {"name": "3. 均衡默认 (T=0.7, top_p=0.9)", "do_sample": True, "temperature": 0.7, "top_p": 0.9, "top_k": 50},
    {"name": "4. 高温发散 (T=1.5, top_p=0.95)", "do_sample": True, "temperature": 1.5, "top_p": 0.95, "top_k": 100},
    {"name": "5. 强重复惩罚 (T=0.7, rep_penalty=1.3)", "do_sample": True, "temperature": 0.7, "top_p": 0.9, "repetition_penalty": 1.3},
]

for cfg in configs:
    print("=" * 60)
    print(f"配置: {cfg['name']}")
    params = {k: v for k, v in cfg.items() if k != "name"}
    print(chat(test_prompt, max_new_tokens=160, **params))

配置: 1. 贪心解码 (确定性输出)
1. **深夜不眠，智享时光**
2. **光影交错，AI 共话未来**
3. **静候微光，唤醒沉睡的灵感**
配置: 2. 低温度保守 (T=0.2, top_p=0.5)
1. **深夜不眠，智享时光**
2. **光影交错，AI 唤醒你的梦境**
3. **静候微光，共赴一场智性夜宴**
配置: 3. 均衡默认 (T=0.7, top_p=0.9)
1. **深夜不熄灯，只为唤醒沉睡的你。**
2. **算法懂你的深夜，咖啡只解你的困。**
3. **光影交错是生活的秘密，香气温暖是城市的晚安。**
配置: 4. 高温发散 (T=1.5, top_p=0.95)
针对一家主打**深夜营业且富有主题化（Cosmetology/Soulmate/Detox/Solitude）体验**的咖啡馆，以下是为你构思的三种不同风格宣命题调：

1.  **浪漫治愈风**
    在午夜降临前，让灵魂与饮品邂逅的那一刻，就是一场关于休息的静谧梦。

2.  **时尚悦己风**
    在这喧嚣褪去后 8 小时里的精致片刻，用你的专属设计，修补好你眼下的裂痕。

3.  **静谧独居风**
    深夜的等待不再仅仅是时光慢走，而是灵魂在独立呼吸的回响与拥抱。
配置: 5. 强重复惩罚 (T=0.7, rep_penalty=1.3)
1. **光影交织，唤醒沉睡的你：今夜别想睡，只想在这里与 AI 共舞。**  
2. **凌晨四点的静谧是艺术的起点：这里没有闹钟打扰你，只有代码流淌在梦里。**  
3. **把烦恼交予算法，只留咖啡留在心头——这是关于“夜”最温柔的宣言。**


In [4]:
user_review = (
    "快递倒是神速，昨晚下单今早就到了，结果拆开一看屏幕直接裂了条大缝！"
    "找在线客服排队半小时连个活人都没有，你们这服务可真贴心啊。"
    "订单号 OD20261122，别废话了直接给我退货退款。"
)

# 1. Zero-shot（零样本：只给指令，不给示例与思考路径）
zero_shot_prompt = f"""请分析以下用户评论，提取订单号、真实情感（正面/负面/中性）、核心问题和用户诉求，并以 JSON 格式输出。

用户评论：{user_review}
"""

# 2. Few-shot（少样本：提供2个标准输入输出示例，规范格式与边界判断）
few_shot_prompt = f"""请参考以下示例，对用户评论进行结构化提取，严格按照 JSON 格式输出，不要输出多余解释。

示例1：
评论：衣服面料很舒服，就是尺码稍微偏小，订单号 OD1001，想换大一码。
输出：{{"order_id": "OD1001", "sentiment": "中性", "issue": "尺码偏小", "request": "换货"}}

示例2：
评论：包装得严严实实，结果打开少发了一个充电头，真是大惊喜呢！订单 OD1002，请尽快补发。
输出：{{"order_id": "OD1002", "sentiment": "负面", "issue": "商品漏发", "request": "补发"}}

现在请处理以下评论：
评论：{user_review}
输出："""

# 3. Chain-of-Thought（思维链：引导模型分步推理，先拆解反讽与事实，再输出最终 JSON）
cot_prompt = f"""请分析以下用户评论并提取结构化信息。请严格按以下四步思考并输出：
第一步【事实拆解】：列出评论中提到的客观事件（物流、商品状态、客服体验、订单号）。
第二步【情感判断】：检查是否存在反讽或“先扬后抑”，据此判定用户的真实情感（正面/负面/中性）。
第三步【诉求定位】：明确用户最核心的售后诉求。
第四步【JSON输出】：汇总以上分析，输出包含 order_id、sentiment、issue、request 四个字段的 JSON。

用户评论：{user_review}
"""

# 统一使用低温度（do_sample=False 贪心解码），排除随机采样干扰，纯粹评估提示词效果
print("================ 1. Zero-shot ================")
print(chat(zero_shot_prompt, do_sample=False, max_new_tokens=200))

print("\n================ 2. Few-shot =================")
print(chat(few_shot_prompt, do_sample=False, max_new_tokens=200))

print("\n================ 3. Chain-of-Thought =========")
print(chat(cot_prompt, do_sample=False, max_new_tokens=350))

================ 1. Zero-shot ================
```json
{
  "订单号": "OD20261122",
  "真实情感": "正面",
  "核心问题": "快递破损",
  "用户诉求": "退货退款"
}
```

================ 2. Few-shot =================
{"order_id": "OD20261122", "sentiment": "负面", "issue": "商品裂开", "request": "退货退款"}

================ 3. Chain-of-Thought =========
```json
{
  "order_id": "OD20261122",
  "sentiment": "负面",
  "issue": "商品破损",
  "request": "退货退款"
}
```


### 5. 模型幻觉（Hallucination）<sup>[11]</sup>是大语言模型当前存在的关键局限性之一。本章介绍了缓解幻觉的方法（如检索增强生成、多步推理、外部工具调用）

- 请选择其中一种，说明其工作原理和适用场景
- 调研前沿的研究和论文，是否还有其他的缓解模型幻觉的方法，它们又有哪些改进和优势？

#### 检索增强生成（RAG）的工作原理与适用场景
* **工作原理**：采用“先检索、后生成”的架构。接收到用户查询后，系统先通过向量化或混合检索（关键词+语义），从外部权威知识库中召回最相关的文档片段（Retrieval）；随后将这些片段作为参考上下文拼接至提示词中（Augmentation）；最后约束大模型严格依据该上下文生成答案（Generation），以非参数化的外部知识弥补模型参数化记忆的不足。
* **适用场景**：
  * **高时效性场景**：如实时新闻、金融市场动态等超出模型训练截止日期的领域。
  * **专业与私有领域**：如医疗、法律咨询及企业内部知识库问答，无需昂贵的模型微调即可快速更新知识。
  * **强溯源需求场景**：需要明确给出信息来源和引用依据的严谨问答任务。

#### 前沿缓解幻觉方法及其改进与优势
除常见方法外，近年前沿研究从**解码策略、自我验证与对齐微调**等维度提出了多种新方案：
* **对比解码与推理期表征干预（如 DoLa、ITI）**
  * **改进**：DoLa（Decoding by Contrasting Layers）在解码时对比模型深层与浅层的词元概率分布，放大深层编码的事实知识权重；ITI（Inference-Time Intervention）在推理时直接沿“真实性方向”微调注意力头的激活值。
  * **优势**：无需外部知识库或重新训练模型，推理额外开销极低，直接从模型内部表征层面抑制事实编造。
* **验证链与采样一致性检测（如 CoVe、SelfCheckGPT）**
  * **改进**：CoVe（Chain-of-Verification）引导模型先生成初稿，再自主拟定核查问题并独立作答验证，最后修正原文；SelfCheckGPT 利用“同一问题多次采样的一致性”识别并过滤幻觉句。
  * **优势**：属于黑盒方法，不依赖外部检索或模型内部权重，能显著提升开放域长文本生成的逻辑自洽性与事实准确度。
* **事实性对齐微调与拒答训练（如 FactTune、R-Tuning）**
  * **改进**：借助自动事实核查指标（如 FActScore）构建正负偏好样本，通过 DPO/RLHF 进行事实性对齐；R-Tuning 则训练模型识别自身知识边界，对未知问题主动拒答。
  * **优势**：从训练源头校准模型置信度，减少模型“过度自信”导致的强行编造，使其具备“知之为知之，不知为不知”的能力。

### 6. 假设你要设计一个论文辅助阅读智能体，它能够帮助研究人员快速阅读并理解学术论文，包括：总结论文研究的核心内容、回答关于论文的问题、提取关键信息、比较多篇不同论文的观点等。请回答：

- 你会选择哪个模型作为智能体设计时的基座模型？选择时需要考虑哪些因素？
- 如何设计提示词来引导模型更好地理解学术论文？学术论文通常很长，可能超过模型的上下文窗口限制，你会如何解决这个问题？
- 学术研究是严谨的，这意味着我们需要确保智能体生成的信息是准确客观忠于原文的。你认为系统中加入哪些设计能够更好的实现这一需求？

#### 基座模型选择与考量因素
* 我会选择以deepseek或者gemini去做第一轮的了解，但是真正深入可能还是依赖chatgpt，此处说要设计论文辅助阅读智能体，理论上好像更倾向于decoder和encoder混合架构的大模型，但是就开源模型而言，我认为deepseek更好或者追求智力就kimi k3.1 pro，deepseek有个harness的个性化优势。
* 我主要会考虑上下文能力，多模态能力，模型自己的智力水平，他的拓展性还有token的费用以及响应速度。

#### 提示词设计与长文本上下文超限解决方案
* **提示词设计策略**：
  * **角色设定与结构化模板**：赋予模型“资深学术审稿人”角色，固定输出框架（研究痛点、核心创新、方法架构、实验结果、局限性）。
  * **分步思维与对比矩阵**：引导模型先定位原文关键论据再归纳总结；多篇对比时，强制要求按“核心假设、方法、数据集、优缺点”等维度输出对比表格。
  * **边界与忠实度约束**：在系统提示词中明确“所有结论必须严格基于提供文本，原文未提及的内容直接回复‘文中未说明’，严禁自行推演”。
* **长上下文超限解决方案**：
  * **结构化分块 + RAG**：按论文天然章节（Abstract、Method、Experiments 等）进行语义切分，结合向量检索与重排（Rerank），按问题动态召回最相关的核心段落。
  * **分层摘要（Map-Reduce）**：先并行对各章节独立生成高密度摘要，再将章节摘要汇总送入模型进行全局总结或跨论文对比。
  * **上下文缓存（Context Caching）**：针对围绕同一篇长论文的多轮高频追问，缓存前置 KV 状态以大幅降低重复计算成本与首字延迟。

#### 保障准确性、客观性与忠于原文的系统设计
* **高质量文档解析**：摒弃粗糙的纯文本提取，引入专业学术 PDF 解析工具（如 MinerU、Nougat），精准还原双栏排版、表格结构与 LaTeX 公式，防止因输入错乱引发理解偏差。
* **强制溯源与原文锚定（Citation Grounding）**：要求智能体生成的每条核心结论必须附带对应的章节号、页码或原文摘录（Exact Quote），并在前端支持点击高亮定位原文，便于研究者一键核验。
* **独立验证环节（LLM-as-a-Judge / 事实验证）**：在结果输出前增加校验节点，利用轻量模型或自然语言推理（NLI）模块比对生成内容与原文的语义蕴含关系，自动拦截无原文支撑的幻觉内容。
* **确定性参数与置信度拒答**：将解码参数 Temperature 设为低值（如 0.1~0.2）以抑制发散性编造；同时设定检索相似度阈值，当原文缺乏足够证据时强制触发拒答机制。